# Robot data quality: controlled Lift policy experiment

五组 × 三个训练随机种子 × 100 次相同初始状态评测。先通过独立 pilot 再冻结步数。BC 使用本项目 PyTorch 实现，数据来自 robomimic，环境为 robosuite 1.5.1。此实验不代表真实 DROID 机器人性能。

运行前在“代码执行程序 → 更改运行时类型”中选择 CPU 和运行时版本 **2026.07**，以固定 Python 3.12。本轮实测使用免费 CPU；复现同一实验请保持同一后端。安装器拒绝在 Python 3.13 上编译旧版 NumPy。

首次安装如果更换了已载入的 NumPy，安装单元会明确要求“重启会话，然后全部运行”。依赖保留在当前云端虚拟机中，重启后顺序运行即可；不要在混合 ABI 的旧内核里继续执行。

训练与安装仅在 Colab；断点保存到本账号云端硬盘项目目录。免费 CPU 不可用时停止并保留状态，不购买算力。此基准固定为 PyTorch 2.11.0+cpu；恢复入口会核对版本与计算后端。

源码包由 `scripts/build_notebooks.py` 从仓库中的可读模块生成，SHA256 为 `31f876b9b0bbab01fca47eb93fc79c6d2d4b18adda616f4432f73c4fa55ec998`。Base64 仅用于在 GitHub 发布前让笔记本自包含，不包含数据、密钥或个人文件。

In [ ]:
import os, sys, subprocess, base64, io, zipfile, hashlib
from pathlib import Path
print('Cloud Python:',sys.version,flush=True)
assert (3,10) <= sys.version_info[:2] <= (3,12), 'Select Colab runtime version 2026.07 (Python 3.12) before installing these pinned dependencies.'
loaded_numpy_version=getattr(sys.modules.get('numpy'),'__version__',None)
os.environ['MUJOCO_GL']='egl'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['PIP_CACHE_DIR']='/content/robot_audit_cache/pip'
os.environ['NUMBA_CACHE_DIR']='/content/robot_audit_cache/numba'
for key in ['PIP_CACHE_DIR','NUMBA_CACHE_DIR']: Path(os.environ[key]).mkdir(parents=True,exist_ok=True)
print('Installing pinned simulation dependencies...',flush=True)
subprocess.run([sys.executable,'-m','pip','install','--only-binary=numpy,scipy,h5py,scikit-learn,mujoco','numpy==1.26.4','robosuite==1.5.1','mujoco==3.3.7','h5py==3.14.0','scikit-learn==1.6.1','requests==2.32.5','PyYAML==6.0.2','imageio==2.37.0','imageio-ffmpeg==0.6.0'],check=True)
from importlib.metadata import version
if loaded_numpy_version and loaded_numpy_version != version('numpy'):
    raise RuntimeError('Dependencies installed. Restart the Colab session, then run all cells to load the pinned NumPy ABI consistently.')
source_root=Path('/content/robot_audit_source');source_root.mkdir(exist_ok=True)
bundle=base64.b64decode('UEsDBBQAAAAIAAAAIVyK95XqUQAAAFYAAAAbAAAAc3JjL3JvYm90X2F1ZGl0L19faW5pdF9fLnB5FcrBCYAwDADAv1OEDFDqAC7hAiE2eRS00SYV3F59HhwirrZZgOhhzaNzVGtwDd5rPMBDagA3AR+lqDucXaWWPyVEnIhu7f6JCBbAnOaUcXoBUEsDBBQAAAAIAAAAIVwdWuNAEAcAAOkQAAAYAAAAc3JjL3JvYm90X2F1ZGl0L3BhdGhzLnB5nVfbcts2EH3XVyDuA6lUoXLpQ0cepWNbcqxOLHkUKZdJMhyIBCVEIMEAoC91/e/dBXiToiZN+WCTwGKxe3b37Oro6OhECJIr+YVFhtwobpgmVDGipDQsJuw2FzziRtwdk3c8i+WNrqSMJGcDJ8vwNIuDo6OjTqJkSsIwKUyhWBgSnuZSGUKzTBpquMx0p1OufdEyq96lrt70pjBcVF+GpXnCBXN6c2o2gq8qpVfw2emcnZxdjMPx9C0ZkvsOgcdbjC+vvAHx8LTXg+/dz6vJVehOjSZz3Mi5XV++3V0urr2eU3hxHl7MLse4uCnWa56tExoxPIM7y1N3bm+7vylWLQXvx4vDYrfMlJpGJ4uTN+PFm8NyMTVUM6NbOl+P57PT2aI2TjAlV9KqW8zmZxf1hpEq2uDy+9Gr0slq6zZeVxovr16fzabnk1clACk1uZAGIMej0+Xl6ckuQlmRrijuTa4+LC5m03KZ53dmI7NK7Z9L2B3PQ6e7OvulACmm+pHMEr5GJZXcfDldTC7H+4KqyAxP2b5WBG1fFKGq5Jxlyzfj+SmAa8NtrXtSaKZsOliBqw/Ws6v5+Hzy3klFNNrgdQ+dTidmCYloJjMeUeFjIg5s/nXJk5f2ZWBvU0xLcQ2VM7TJGpTfftfuGnZrYEcb5VeCbuOXurpSegdKoHgyYjYMCtCwLGbxE8GytdkQnTMhICFIIhWhJOVa41e04SImNxusFKcPTIXzioDSTEgak0gxinXLjQbTFMtMQKZSpVTwv1ijlyaAX+kGfJfavhRZZIv3GKziGrSCJrgBBAW9tZYiY/Rjxa8ZWMOibWCP8gRKO8hoyshwCPliPLAstkAE2lBl9A03G9/7BM8fnz55XbuNmv9NZDk9QzkHt4PcooUx8PHQx98Gn7ud1k4FdRlFxb4WHLgpErKIw1RCVvlofCueU5mxQen6CYSLCkYwTyEWlY8xqIigqu4wK4gu1DWuUpJQiEFMRih0/obAoYxZ6IJS36g+yG65Bo0RHBNwIUkLbUjGrgH/FaxFEcuRgynAXSi6EiWwueQZ0CSooGtWo4weBFSHudT81u/uILdrer8FsgVtb9/rBlyXuLRhplwzMnc1OFZKKt97JeUazLLOEu4ywh5k8THgnBQ2OaFT2J7RMt+2D6RjcAKQeHtZ++N1IUyRoFqTd1JtdQ7M54zA2IUhz7gJQ18zkfSICxuUE/nbugL/MHRQYvivR74W0HTC9WpAEqgBLL0XT1suoZIAdcBGU9sWE7sKJQbJy7JrriB+a2Z8z9HtyXI0WYTz2WwB/IHyQXQT+114at3/kvj1lYHFOijynCm/Sx6BzGjgNbY1kL+loqgAr1jiUMdmgAPHPozYjgaI5AFjHrWMwXAhF9VG7aRNKy+QJfsmzd3LRqbMvd1UEXKfKYp2uz9yYgnfUDPVbEFsIfbPJqRW17OmUaLvoEDSptbaPh0q48aR3Ri7PFjdIVZDAvnnV5lBHpNnT5//9vjxC8i7KsuQuasMA34zEKh2lu1R/k8Yg4p3cq3JwH591U7YEAfbSKAkK4HQyJbyH6CdeLYwmI5ozprEcZVzX6l8OK5ZcniP9z3SuARCw/v6Klg7nFPtBLfG/q/cnlet05GFxQo4RRZG85jtZXRJ7bmdAKu4wXBU5BaaPRbHBxsmWmtjCjlA6rkxgOtS7e8hCYUAFY/dGv23KeHYMnBjQdeFrHvgUJBuIWV912X1cKEKuNWyfSi39nP3VEMyH9HCz+WI4JQ1oi2xcmYZzaaLd/PJYnwKk9DZbDT28Kz3zPvOmeXi/PdvxKoxO8AXsL20oHG99LpvR+gWzSGqJZ6+h1NXX9EbJAP7DtkWMa1ZjCtALBzmWJxfiacYTvC6D7XIVEYFrgm51t5eFBoLEOv/AmydDYWGftJkA9z0zcygi9TPkfOM7ZihxlEIXcrRoYar1VrIle89hqBD2udYjIgWMH5znW1uoeWVkjtoHHPs/FQM8HbA9Km1JAaWbkyBKTSussyZ3EJXMWxm7idREHO9DRuvSsZGmXZRWn2/ti4nL78hwUOluNPZE+/FU/KKn9Z8YQ8PrPLhPf596FnSg6aDK81texTRMuNX8rxhWzAKDf+hId4k00WSQJ+AiDs8bIc4JlvGctCIRm4YjQGM9ABB3HuIkjfY63OQbeiEQ8NzfmEG8hSmi2pxH7XejrH4eGhQLY8foMQWSog5i4v328H+ELHt2hzb7pDQw0OTSvjD+Dv9p0eukTYPcFzZX3aKZq+nWH4u5/+foClo/tVPGhwOQhuUW/s7KHDvEF8vMMgO7UOBJfMQ53If3QriIs21bx3oAQAxXD18DtdCmKF/Uh1xPjynQsMuQEELYYbgORSTEPImzGjmdrt4JJIxzJdDrzDJk9/37gWCEZAn1sJu5x9QSwMEFAAAAAgAAAAhXDYGdoZbBgAAoRAAABsAAABzcmMvcm9ib3RfYXVkaXQvZG93bmxvYWQucHmVV1tv2zYUfs+v4PwwUZkjO0ZadBrUoUMLbMC2Bm2BPgSBQEtHMRfdRlKJM8P/feeQlCw7di96kUQenut3LpxMJteyriFnbbcsZXahm05lwPLmsS4bkespU6C7SixLYK1QRoqSFbIEzUSds6ypDdSGrYRegY4mk8lZoZqKpWnRmU5BmjJZtY0ySF03RhjZ1PrszK/RqVIu+99/dFP330ZW4FihiKxTCqVEjqfuWX5aKRD5ddOU79aQdaZRUyZ0mjVVW4KB3J1vhSEh/aFr/HUbnSpxPUKjNPS7/3aNgUE/Bf92oA0qbA9ExGqQ/rlR97oVGZKf5VAwvRKLFy850cRWSsguXjNtVHzG8FmxpLc38qSh3XiUZmWVjJoWah6oZRCiGUhc5yW4w/QUjWLZqqvvmayZNKB4KaplLmJPGZEz+BU7Z5fzxdX5+SKcsmUQhDsOVo2oa3NhgFtWTgMF6NYat1awzuUdWoyqOaOMAuAK2iYmSwgLD1JjDP1v0ZQ5KPtjrS2lNje5zMytk4ouRrOLYGVMq+PZbNXd3cn6rkCvRVkzE62coTJCg9GzDYnZzkgifTs529nGydj+qjDESssHSIzq4MdSVtIkl/P5PHBGNI8aZd3cOqeuEKIkfmc9AqdF8AES9XGN7sBwJJpauDWdSfjliyn7eR6Gz45FSkgNKUYh1YjjTvMRDcqOYI2ZkPO1DdSagjQcJWDzkMmCrW8C89RCcMuShAWUR8GOjfPWcKqU9b1TMaiReTBlm23o/pEy2IsdaeBDVoDJVimx5o863sGUgncqjphc6ilmFDkbR/qIPX/Sye7fBITSwDk4R//hBtpNi5zwzgOK5UyJRwTwzEqzr9JpCusWMsxKPCVrwy1LZ42W/wFa13u9J0wxTXrZjrAs9NgLjcy9F7TMIRMKqUmviFIq1V1RyDW3C+6b/cSC6AGULCTkNij+OAbGksEa8YuBtaXN89xfdNww/jzEV0qaUyR7lXdoq8CQ8iQksnWU9/woTVODAUUXZE2O6ZAEnSkuXgUj1KFGxMHZ2QcL3ZokQ+isPjsi68TwuDIjoNCBIYBU+6iuVve5VNz96OQTphdCgsxOm3v76xTry/83eJlIvXO/UgMO8x/h35QP+yXAVmUsQ5ioWhSQTGaTcOvSnpJNGANVa2zKifoO+ItR0SNg7zmiKQqUhyp5cw7Dia7vd4bQQ4l1Y77HBsk8p9cnPG69TkWDfehqKjDvlGoUL4LrURvFsxlArnfpQUrEDM0vt6PasC/yh1NRpmeFAANFxXATfCB/BDG6f/lkQCcbx2B7EWxH7Kx5m+0zTrY1PS+WXkDi31OqISAqD5yhlC7mU3a5wKymbtZXtefqOnB+vcge9wQlwXDancEBIAdy0WL+8ri4PRjMj5L4ruxw4BqzWAaHPgseTzTrw2eveQ/6UhdP/QjFF6POfZoRPb7fPyq538dPPai0P2KgLBHOX0Ls4XMUwW/9dOjRC/k3wvdEyn0Rzk6B9x972X/U/YA3DKnH5blRBiM8TGZW+DOl9vqNLfLu4Egr2vrG5P74+xsUxiqpK4GN2Kv2C2sx6qAeRkM0QkLWmgRglYsOlO89hUWxxEppC+w+he8wm4DqJqY4vaZs1yzioVPgqu3ctILp61pFPBhHK9ZDQexNnx5FRdC1LtHTgXzsn/3y0Xc7C1LX7mwvzLuq1Zx0n6L1OQI/oUH1WSv8Yu+ih5CHNZ8P5emD+3hn163VHjMH2YQR7xsGNsurE2HdW6X4RroEaHkla744P/ccpuyVb9pHsLDLkkJgpadZZTyhVaJ++r4JTQJS70bsKXvEk1iAYxqoEApXR2dwkeeS/GEbN97keCXWfD61QxiOoxYMt3iS8a8Pc+t+BgwPUvh4GUFXfxfPg36LnWOYpb39/sqko2wF2X1Kg4HgOwv7mVh3pRldBqiWP78skiNS78LEv20xb5FodO/yl05MNdrAKWeJNw++G7MxDnrq02+XcnZsDeP90dnaY1dGNu0Sh3adONoeX2W51yI8HOqyRtFE7bYjZzk/zB5ajETb0v3EHTkoNorAgGX9/ee//3z/5i3blNjw3LkQ5y/66wOwpYJGLG4mZNPkdrRASJjczi7hZRxdFlv21284rhdlp1ejKdIns8Y7NOReyJTdw1Pi7rNsHY8wcfY/UEsDBBQAAAAIAAAAIVyWTFg5qRkAAONQAAAZAAAAc3JjL3JvYm90X2F1ZGl0L3BvbGljeS5webVceXPctpL/X58C69Rbks6IOhJtknEmVXqyvet6PlS2k6pXU1MsisRoGPEKQUoaq+a7bx8ACXJIWclmVT54AI1Go49fN0A9e/bsosjrqkhTGQuVZE0a1kmRC3lfyirJZF7PRCzT5EpWYS3TrVCyDPFSrKsiEy8/fnjzUkRpmGTKPzj4VUklquKqyJIsicT/vHx9Br2zIld1RXSVuEvqjag3MqmooWoSoHXin/knQua3SVXkOKiIinydXPsH7wtx8evL85m43H4uqmgzE0Vl+ISrRIkkK4uqBu6bvE5SEQI3RRMLGDDJk/z6SN6GKZDLsjCP/YNnz54dEOdBsG7qppJBoCmIMM+Lmrk8ONDPoqLcmutNqDYgCHP7uypyc10oc1WDzHiAMqyxuaF+Cbct2c1ZRzZvsnIrQiXykjuqm1SGVe7LXMnsKpWGwhtV8OK8Liqp6n7jspJlVURSKZiz6fGxuGpU/SkKU1kdcHs/Lu7ytAhj02Yt62gTrJNUzoTahKdn/zUD2Ul5cPDhn5+Cf7369yexEK5TXP0uo9qZCQdXrT4OpFwHZaEGT/5oQrvRdZWUoEfBH9jSO7j48P7lm89vPrxnmhGwnmNrWOw6zJIc1Cqm3rBURRbEVVHibdjESd3eFVUIHfnWOzg4iOValEWaRFscPU3qbbCWIS6tcsMI5TUH0fp5HFZVuPXE4S/W7fxAwA8oeB0CS/A8TtZr3W0mwvtELY49avO7rG6sJtSl3yILr/OkbmLJzVKYT3rt50WV9VqfcOtKAos5NiRG3CVcZSAPF59cGdY9b4ZNVB2bB3Sfhfd77Yjq3o+h2jLHBEBSaC2yez4T/k9nHXWr/eOENRsoHs3syCzEL0D9xylSz3szbIW6As9Tb0u5WIO+1mapLV3RHdRcxElUL8HHzKylhe7rilss/BNQbinjxfenHi95BVZCy8S65gPpsEnrAJ672JIX6UZuFTRT5GDMcPwqKsDfwDsUFlBP8tqt4FHspjJ3sZ8nnrcMeB53UjIFKwJftUAG/GhTJJGk1jMmOAO9KNMwkovXYaqk59eg2ap2zZidnQCJh5u5CH10Ua4n1uANb0B2wInQjPrgWjPlejvqXFcNeN6FWK7oFtsnM5wh9pDghcjDu4ZFLSf8KarkGpUZOmvKS+i2at8na5GIf4hTsViI43lvjZMclkaq/nxRQoYmqIyWYO/p0RHqYl8WPcK2KIibpR5q1R9reXjizyC+gDao5Itc0OC6qTdrp+aD6yvl8mTl9Ye5gaZA0Dk9DsCPRRCZAhZBoMrkRiqnbS2BxfnjLGqFg1jbTnQmzvpOZDjwOrmVgaplGURJFUHUqwwDWaLA211TuOzYoFX2Q/C6oIoPjiwTVcTSmeNCg/NEsngD/00YIzpY5q2dKUUF6KUDoM8P2jmAjl5ta/C2nudv5H2cXEvU2En6UVFVTQlC+doAewKcGGnX86h2rxnLQ7sOVu1ABwtoB2EFtdmlQKS5UY+6jRHisEq2T5qgxYTGvEqvQ9+3oK7SGPz4G/F5IyFY1TALsN5KRhK0QxmjFEWebn1qk+QYrRHHMYsAkwDbAMIRYXXdEMACAqHQgdIn8vesn6oOoxt3ORVS+4uy0o4HfQhOa+W1pGzw4Xr+GkI4YLJcrTEg3uuQCbqJbmUAbdw8gH+TDCGeWnx3fDyzRIwL00lVQwVgupYLFDGEgOD34kotTmlMM5KKkDIMdUhj+nQfqDArU5iUbtQBDfKuMJ9lsmJPiROkWH2Ny+YyuRkZ0sKB0QGnOd7ykFZuvmKPa8EYXG5Zu9PxRnwrTrw/ExCYYwsMIcvVsrV45rtCvkkHdkZ/tdJjc43AOH4OFHEIy+Z93R8x7h5sm1N0ujVh6Rb5sAmY2ITB44aUExpYBHajI1hI8C8N0PUfp29jy780gEVgt9PYks01gHVEvI1iZ/MEB6gXS5F0abUdylvsFy2emHKoIHhZQyegh1i7XWI9A+x/6w2nobWghQjTxFlm2mmju4ZUI0oUD4DELav5T1sA3hHP6emUgXCYpv9XslqLpujaVvnnCGOjkkzhoa+JBh+2z7zZ0Bp0E+uhNxvqm25js6SVqAttet107JkN1cvkQxVm6dK9IzeCmbSOXyVHK3iwdNjDO4zjICRUWwwA8h58JmnLPfsOKd1y6SB5B0AUXd7y8sOtcwup+1GarOujcuOQLdwDZch1wQEBHITmmFo6K5Mi5ckaXDsM1OWdxOX0EMQZd0e60PVO+XjlYkrtOnFYh0dVeAejH3Vk+NoeG2KCrG5DLT4TM8EcdzP4S02oNoG5uf8a+cJBcCGBMqTnG1i61IJ4OC5058dLYsPpMLHMb4NMUgssFPiYcisXG/lhXVcKHDW0wHBi+MMfNv4+PriHWBGBL9AYn1bFDJqF6uaIOjmr5dwidAtRO/6zhKjTgFCypsDVMeaB0eCTbgSvD3uhpZLitzBt5KuqKirXuSRF6xeChCoBV4BBhjfhNYTOlgT5KU5KLGl8a82oP5y9rBbKBiWPwEsTJFui2DlJcKA5zPAGJ9khF1PpWLUpuh8qTD0xYFP2+d1pH57beRAMaA2g35AYp6iM2HEbHSZBKmV7ZliL+U5KO7Idy+8ZmIRmZKwewYZROWNITljVyRraqiPtFDw/u4mTykUvktdq8blqJJgiCKcOihu6NZj0JdoBZdyARCtJkRDLbUmNQVK8ycUFgLsrrPmBXlQyA3aJb8CpXKq7K6obVQLGYRyqC1NlEt2ksrPLSWaPtLeL/fImBc4LyH1c5+5q3GyZrB83WQn5ka094IH7PqJdy1bwDIr0ijAkMgs5HjWcbnWgeXeDnrNVaHjT3cAb4zvgubmEpeWJtAuHXmVEFi3yDzgTxGaO1rJBX1Y50zNgKdLcTa+H58+5ETBF/m3eevDhfHk61HsIXixPgSGvE7GZPAFaY4N7MMexysKWRIAUgPASU1tH14MBRb+FSPRCqG0O2lUnkVgnOSCcw1v0RrFeRdFZiHJ07lhWWMBxLj+8fXPxb3H58dXl+cdzrFSKy/NPn0AU67RRG633HGMDdOkQu7T7+3u0tprQWg0AtPLSyEYfNDe5rNGO3CQvGwAhSWZiHF73eayxkG7jCnrg57n/Sf7RgJiTMHXbZ28BWYaVTRdyc1jJtsFH+fbX6XS//RkSpErz30jKmq1F8XOYbzBZ0lLKwhsZgAq5Rvqcl6P/Ba2Ig9skloVOtXoya/cqOIO/w8hNWArAQixlSVU4Q3TZmfCKr7mD8bvfkJ7qXY+4kJydy/uyAA/5rvm9iIpX+a2jiDeMhrCyMfrMpFbocyCINgj6fIsZvyljDHabUAEWzGNZyYongq5DBcV6rSIAc3n31p7zmMwbJYMoxLpggAl1Swz6QPIra/OkJehkxLuO5l2yi9Owqqpa61qR+rgqjwgvBx4whD1/zlNtV5O4CJAw0IeWusgyt6olZZJDYqRFTbsSAngSlFJTHYRLAJXC2Yr3TXa5Bblfp8VVmIqP7//bf3wqMKiP1cYrAP7XMscSKiwLVSMeLy+P9bCFQ4Rxeq6ZbbvpFaisuBnD9lqAMHDrm7j2UYcELYbAH66StGABqoBaaQSLhTiqlljFYqAuQ5OmX0v31MJ9wC2OOmZdSLXDTlFTIaDoKJM5V9s+psPhSPXbwYiMpgZg8LsB5sQfUFHkYUIn9prfchFtHywCnaXe7jqkZYFkglJ8SGfMNhhVe3HEHhJrYeT+aKGCqFfrQbnYjPTck59mHikkcJEojlYutwIMmqYuhsxP7XanFTypFoH4TmSJykJIppy9UbW0TTl4UF7FcpbizSfUNtAv/xpFh3PGgh0MCTJBDngbxvl5/aPjTdRg96dMNGtZEvptYCr4oHPSM+Efn4nnWq2s/musKacDjcC+ABaVdLuGrKRmcnquupDH4uYWy2NKRfXNCWatn0g9BIobYp2gKStIUJCSANkCHECNL1LAp0AG8ElRbZ1H4ZOFUthAWxCF5cG6QaDjaDDRA0EgC3gFitCbsIPb1myZBiFNWAJqiJ4Ir12/nk7zH9Bmmcs4QIgS4H6mrKytDTPfuUC800dIn968+/Xt+ecPH8XHV59efRbn71+K8wtCSp/effjXq2m8xOD3KuJ0v3UQLYQ2kRh1Rk0BFo7raOF5kwX1ppKYVuu0il8CQm206CxHPenBwbS5X9TEIdhgEN6GSYr1XNdyMlYTi36AFtqR4kZXYXQDaqSwNQCQK5lHmyykzWOKl1ZTDLBYwKkyWEAFeBUIXhcVLEqm3C7HiuVtEmE84V586zrIjfMo/+ymnKhsnLGdhy7iWknNatneWMX8oZPsuvYyqFXPH1obAttHaUwMP0XsG/G+qDLIKr6wJ6SDDe0mpTn40UvmaG8EtDK5pzSgKhQkq2lqV0J19CVTiORfnXPXyMr8Vqb8JUNU85r2f81IvF+u9wBn1nPaEufH4PuS0vWPT7oV4UUHYAfswXq67r04pAG8I+hofDa30sWHmdakBf/XrcyQ1vbp3c0OTpt/3LdbqTOx7bZVIWy4dr8C8q8s+SKrdnh64p/HYebyFg1moxlah8IYlFYL/9gIoA5x937tPLTrtyODfMB/duyoo42MbsoiIbRh0rD1fh72AMR2flk7fbB0bLxDR8en+oey/UKJlcqiUe0sCHh1XXB3uwQ0FvG+FUtgJu5kcr2pAZSDVg53t3nySEe7c9qcMQMtHXpvFw1bUT7SqW1jdzRT7Zqh73V6G/vc5hf2yl8r9F10Ik+UuJJbWB2q81SYVypM0NkCWxu9amLAHFb5r3PwAJU1Fun4o7f4xln54NVs2AG8snR91FwCa+wgCe5rCghw2HESEc9sjL4v8sHmveVTp5hpyfR4IYERCMejYH5WAP0iTyINXADDqIDKEdACh23xNUp4BPB2wZBm2Z6nYAYxpmFcxjrLPViJe3L648wbtVStrqgY6DQXrGjufXtwomuGTLZDQAxbNzl1ClM/g5CFr92OFNj5CI1OK+FvEVxXIUbdOqiLIIdpWyVEMyJFTkjwYneMCkHJ3nK7+OjbE0/8Q5wcHx/T6ROhBfntCd6OKG1P/uTYXBobwnAYbQD2DtRKLymlcw/a9ubaRi1Lo8pWa2PzHt+9RmRhc83jZNGwVfO5XoTrng7S5qzRvrmtq4N201ZByAD1b4qHjHeGOWQB2zFxHRPY1JUgeGJ7e6cD9/Cm8/1TI9gblRYK5Ar1XKNBHUZxjRAHt6s3RTQEUSS4JdZUVBp89/YSi0yH8FdgdWkGsSPfmHruC/Hu06sXAoOOCTAzcYXJ1AIMydn1XYJENBrSNpkVFAhDq2YNns11/DorHW/Ek6jwloy6Bu/fEvLG6ft6y9+KI/2WjMXXzueP52/eCwpgpFGLB9ar3RFdqB0JbPHQiW3un613sPhhqWS8eBi6qEPtveb+yXqnjBd54P93A1iPnJiNyZZRDfYxe2lQBQdg3wqMBKC1m9Ob3xsAcV8gTH6PJz6+VpHrkgJjn0+JwASGpwPwHpwh2ktL51czoZ9Z6r6yeu9FYd3cxO0XuhmKyG2rSFPlE0tItkA0iGpqYKsrpjAssnQTtCDTKYR6Cgyik2bqgTl+hjfPVruAUtGHjpUdp7aad8RECZ2mtDc6YTwfkzOAlPdYHmBPpDoMxR5IM/4N2KZqMmn0BgMULg1lN7hP1AELCpNU75BgvlTA0igCWEH8in2TXG9H4HF0oJzWGt9b3C6r7oRM+xjGqxgkGIjCIgGoQPGEVpJREnN+tSXgaQ7fmKTdWc3FCH1zEtOqbmCbtpsV/62F764h4SdTGZS/knWfhGZqpEZGCmNKJrrZsu2LyollpSRv5HCTdby21nbtOyhTWKEk6i8UmcaIBVjNEIvhIUH9dqIs1Y/jTYQH5HuJuPkhq+qXJc3q9IGZ9lEjBcj/14ri1zak8Yc/rSAnmOQ6iwwy2vHf5xZ/wj4U1NkeZNMBfZTg6kokJJbGhyEoWGGGqQ0Bt5I9n1vvMwTTmYmA/8AodkWQElpziO8Qou7JSAkRrdFyegTl6ZgwqbEnfhan9IxWaPwAcm99jeIbTeSNC3dDwWBBu0h3SVxv+FJvfeDuw8IJr2Ve3ybyzvGW8/nhyUiNF60QCAfkrAKtbFOi73QRg+lokyvwnzdDO6d57JPUMTHJgM+k8G9PcRNR34005lZZkhEs+WpUeGjNfOdn5fcOWCtvGol1qRYng6PLvGR8vtCCeFpnukcrg+7Iy2rQpxWLXGjvPXnhed8FTx4x7vzwvHNRCC1Z6vD0qihSV98aXK46YD5ZUrWdUR8e9n0rC6EvGFM61sERT7zeSkiIcAGqoqjxPDpr9nDVbaXHbGdC0Q0ufPXb+VsNCx+srgALKXrsjPZJwIYQdt0KYj0/eWYdGzW9RmDfXo1+WJ/XuJBJmENpGKufdiTtxfRmli5O08GtNnemg2HA8mMHWnBPD2TDp1xRH86ea2RTLp/tbYQB7tEox9qEL+5UH+bgYD2Y0wnkvuQjnkRTH//lCFKO7rvR2Z2R5xDy2xftwYpVv/BBxxqBOU8fuRu2Jif5AC16CIVXurij2Aa9d9i7z3dfxRhm0Zk5ojUwzn2CY2qsiXjiPxbiZF+F94tK78YqRopqrQT+Osg4wA0aFS5MF78sykGQqjhx4DqA+XRpSZMzrmJvVsNPRLRCLnmUFfk94034Wed4AhwQnleUBzqDgzK0hPC4xO2rWDcTvyyE/4M5BoogmcTdViitCuT0WRNL6btKZ7Lu6mL7lc29LGufAXQ2Vs5liD054zJu3GvtWC8ZSHpZLrUU2TL09fPT3t13q7ltB7r7zzaf+9/k9NBtr0o8sUXVfqh4NtNDeD2JpNj3CRnvVw2/s1h6qKGmXbIdV9j7obrS2U5mzVbXyaNfOLq9X/kE7d2zXFubR2fkYBU4qdC39On5P4wr/cgYnfyg5dnMivRanPMJKe+sfeO/z2BJwo+a7FfFbnZbjfiXhj06WmqdhtWbdvq1ZymFCfuXb95++Gxi++IBWZn73613elaLB/5/LKBrA+q475vN9HlDyBa+SPwmoKiLqEitk4OlhapasfKUW2nRsdJREPdYIW/GBTurWiew7I21PIfqhykfgcRuhFR41+GQ3FZnkM7EyC1CDQDBUTWXtt+7j5e93RDwIthhD03h6yN+ip61uyLoPcQ/L3S8WodJKuMXIi5o66Fq8KsR9KWyArdR63Ox0AWjXVRk4EgSRBztx87g2jJ49mXkCJA5hAjxHvA//CnjSYylIwrC1MFJnyejKUcjKCy/H+/DJabtU1kmdh+DTW1AIolwP4vQiFDfF8hqBmvLLNmVI13x0WFujbuIuMsV+3hU+TXeanhuf26HMqCm/nVVNOXV1rVzlb1UBTJy/J60hbvs2dCXuNFMKD5iH6EUrY/eqZpAyMVsD/MBD/29mD5rz+xACh/Le4JJLXXort/i41KjOQsV+mG+tYP4mORes9gseSU5qlkqwV2YsQR+d2p/LXYv8LCFFAxn73HHxZyPUb5jjoiwE+hj5GlUMuo7vH08reXCq9MHnSSfB0OhBQa7J4mA2Ejw4A+bIiR5MHO0P+ZsYpvyMWjfMtId37Hi/CNvEeb3X++B/Z4YusIbSmAKtT9t/XkefAyID1NYMjBcdaCxbyVWRuH5Y0mznzd58keD1T780nsI/MdYfDM4l8Wc6WMbnVJCrA/JLjovaSq/tu37dRFE6rZTxEEYbtUYGmFpgwzP3hQok9uibt0D3bncaDnwCjYg4Y8u0ibL1aLnSWgjBp4ZyNbW9vTvOSCpPPLrCca+5Lgqiho/rSktV27Uc+hsdMX9AqWJwgMogIcs5nQgprWJzhgU+k9SU4XCgfYx2oRejtZGiLrfbcgoXf2l2q8GLwPd9Y1meF0KEFgnTCGqWM6MT+rG/SKtfhi0Q1rf/vMz/s0CfG39ToE+9umdOLUo4Jw0Abwc9u+jJM2LKQGRooATjJZUW+erHrsWhusWsO0e6yqyqzt5jPa9bsWHUa0jMtgfetj7VsLssMyE+Rh1/+sIavCCXCP/nh8daktZUezNI7mHoazDjiao8O+DYRvTwJn0gHEzJQQ9v2GdlBwS3ysH7nl+bYHs2QKTQdkN26xqjzphMNTNTNabAvGp5k/Ud8XhXbg1ttKt1nyg/1yO3lDlqn+69IVAdUZswt+c708OP97Vmc2fBCLsSLR68LHxgByUi+eueBcSZi3zeuFo5OPsf2fT+7LpIcKvXAdZEKedpNfLaGXU8SufazhR8tMZdF5ybxQddm5/A41/fHqGh1inXv/0A7xe7cYA1d5Xy/qz3AxSfBXYH2ub394RXsuAwL9qpwNI5rm21e4D3kN+YJPwpuY7PSjOPCjLbvI0Fk7RHoru+yMNhfO4hEd//uqALO5H56g/Yv5rIrW/gDYS1V/fTqTJZgNXp5Z2iZwzXwKb+EGf0pVujuGg7afefqrb/9Tr9ZvfXh22GiXOLy5eXX4+f3/xyEFmzRZ+x2hlYHQs4FrqXxhCJ2s6qEL3Jkdbc0va/dOFIqd3onD46TajzrTXkY6YW90e+TpkrDulovaow9r86Jgm8bTHHUtGxzpDnmt10+Dyb04UtHgxYHGXZb/gsOrH6zHoeQktTaKeVMo+mji5JUETMt7JtQ4w0/FRc8c7vFYVYeSbFyZAcIHhPoaR7ogikzBT64G6kW3Gp5Q2MVedtRR1DB35luPrxU1DYyzx6b2zS4LmxUilsyc6kAHNn6KrFsr+mc1xXfxfUEsDBBQAAAAIAAAAIVwJRXXydAEAAJUCAAAUAAAAY29uZmlncy9wcm9qZWN0LnlhbWx1UluO3CAQ/PcpOMBqDBhsM1eJItTmMUOEBwLY2d3TBzyjrBMpkj/ormrKVHU2Rl8Ro11Mxrp3aROsJl/RgLufWyggb8tR7E6bIL9ac5dhjd5cO4TyppTJdYhQjHFtWHB+S6YN1jqHLVX8zKxHhJKJ4Yp+PMLtBr1OwWn5wl/w7rILj3qtBTXieZoFZ6BHOxo8croIRQhwPI7tU5TiY8zDYnydOf3G/9Re+D9qjA7EcKFALMLCLJZ5GpTVjGjOBWVQ3zTMg4KzGu5i8E59NKmnUApLWN3qVP/nJDUUyKbk7iw3MQuYzGwklk1CaWE14UsTIiPBA66eipHiob3HVb/RTi68986WPt57H35J7Va5E365a8srS4WUtljq5S1MVQ4VfGmGlATuIXONvIb1jUxviIo3xIbvFTM7+Cckc4FUqoX4meaBmOhy0G0zyNGMzofyN1/gE/LFP3LJxcT83IdWLlDUXWb3adrSzLVzD8l9HvZXwm9QSwECFAMUAAAACAAAACFciveV6lEAAABWAAAAGwAAAAAAAAAAAAAApIEAAAAAc3JjL3JvYm90X2F1ZGl0L19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAAAhXB1a40AQBwAA6RAAABgAAAAAAAAAAAAAAKSBigAAAHNyYy9yb2JvdF9hdWRpdC9wYXRocy5weVBLAQIUAxQAAAAIAAAAIVw2BnaGWwYAAKEQAAAbAAAAAAAAAAAAAACkgdAHAABzcmMvcm9ib3RfYXVkaXQvZG93bmxvYWQucHlQSwECFAMUAAAACAAAACFclkxYOakZAADjUAAAGQAAAAAAAAAAAAAApIFkDgAAc3JjL3JvYm90X2F1ZGl0L3BvbGljeS5weVBLAQIUAxQAAAAIAAAAIVwJRXXydAEAAJUCAAAUAAAAAAAAAAAAAACkgUQoAABjb25maWdzL3Byb2plY3QueWFtbFBLBQYAAAAABQAFAGEBAADqKQAAAAA=')
assert hashlib.sha256(bundle).hexdigest()=='31f876b9b0bbab01fca47eb93fc79c6d2d4b18adda616f4432f73c4fa55ec998'
zipfile.ZipFile(io.BytesIO(bundle)).extractall(source_root)
sys.path.insert(0,str(source_root/'src'))
print('Source bundle verified; importing simulator...',flush=True)
import torch, robosuite, mujoco, yaml, importlib
import robot_audit.policy as policy_module
importlib.reload(policy_module)
print({'python':sys.version,'torch':torch.__version__,'robosuite':robosuite.__version__,'mujoco':mujoco.__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})
assert torch.__version__ == '2.11.0+cpu' and not torch.cuda.is_available(), 'This benchmark was frozen on PyTorch 2.11.0+cpu. Use the documented CPU runtime; do not mix backends or PyTorch versions when resuming its checkpoints.'


In [ ]:
from google.colab import drive
if not Path('/content/drive').is_mount():
    drive.mount('/content/drive',timeout_ms=300000)
assert Path('/content/drive').is_mount(), 'DriveFS disconnected; do not write checkpoints to temporary VM storage.'
import robot_audit.paths as paths_module
importlib.reload(paths_module)
Workspace=paths_module.Workspace
cloud_root=Path('/content/drive/MyDrive/robot-data-quality-success-prediction')
cloud_root.mkdir(exist_ok=True)
ws=Workspace(cloud_root);ws.setup()
config=yaml.safe_load((source_root/'configs/project.yaml').read_text())
from robot_audit.policy import policy_main
manifest={'bundle_sha256':hashlib.sha256(bundle).hexdigest(),
    'source_sha256':{str(p.relative_to(source_root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in source_root.rglob('*.py')},
    'python':sys.version,'torch':torch.__version__,'device':'cuda' if torch.cuda.is_available() else 'cpu'}
manifest_path=ws.path('reports/cloud_source_manifest.json')
if manifest_path.exists():
    import json
    previous=json.loads(manifest_path.read_text())
    history=previous.get('previous_sources',[])
    if (previous['bundle_sha256'],previous['source_sha256']) != (manifest['bundle_sha256'],manifest['source_sha256']):
        history.append({key:value for key,value in previous.items() if key!='previous_sources'})
    if history: manifest['previous_sources']=history
ws.json('reports/cloud_source_manifest.json',manifest)
ws.path('reports/cloud_requirements.txt').write_text(subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True).stdout)
print('CLOUD STORAGE PREFLIGHT PASS',cloud_root)


In [ ]:
policy_main(ws, config, 'prepare')
policy_main(ws, config, 'smoke')
policy_main(ws, config, 'pilot')

In [ ]:
policy_main(ws, config, 'run')

In [ ]:
policy_main(ws, config, 'summarize')
print('COLAB SEQUENTIAL RUN ACCEPTANCE PASS')
print(ws.path('reports/policy_results.json').read_text())